# GeoRoad Inspector — YOLOv8 Training on RDD2022

Fine-tunes YOLOv8s on the Road Damage Dataset 2022 (RDD2022) for 4 damage classes:
- D00: Longitudinal Crack
- D10: Transverse Crack
- D20: Alligator Crack
- D40: Pothole

**Runtime → Change → T4 GPU (free)**

Expected training time: ~15–25 minutes on T4 GPU for 100 epochs.

At the end you will download `road_damage_best.pt` to replace `backend/models/road_damage_best.pt`.

## Step 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics requests tqdm -q
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## Step 3 — Download RDD2022 dataset

Downloads Japan + India + Czech + USA country data (~2GB total). These are the most well-annotated subsets.

In [ ]:
import os
import zipfile
import requests
from tqdm import tqdm

os.makedirs('/content/rdd2022', exist_ok=True)

COUNTRY_URLS = {
    'Japan': 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Japan.zip',
    'India': 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip',
    'Czech': 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Czech.zip',
    'United_States': 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_United_States.zip',
}

def download_file(url, dest):
    if os.path.exists(dest):
        print(f'Already downloaded: {dest}')
        return
    print(f'Downloading {os.path.basename(dest)}...')
    r = requests.get(url, stream=True)
    total = int(r.headers.get('content-length', 0))
    with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))

for country, url in COUNTRY_URLS.items():
    zip_path = f'/content/rdd2022/{country}.zip'
    download_file(url, zip_path)
    print(f'Extracting {country}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/rdd2022/')
    print(f'  Done.')

print('\nAll datasets downloaded and extracted.')

## Step 4 — Convert Pascal VOC annotations to YOLO format

RDD2022 uses Pascal VOC XML format. YOLO requires `.txt` files with normalized `[class cx cy w h]`.

In [ ]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

# RDD2022 class mapping — only keep D00/D10/D20/D40
CLASS_MAP = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}

os.makedirs('/content/dataset/images/train', exist_ok=True)
os.makedirs('/content/dataset/images/val', exist_ok=True)
os.makedirs('/content/dataset/labels/train', exist_ok=True)
os.makedirs('/content/dataset/labels/val', exist_ok=True)

def convert_voc_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in CLASS_MAP:
            continue
        cls = CLASS_MAP[name]
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        lines.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines

all_samples = []

for country in ['Japan', 'India', 'Czech', 'United_States']:
    img_dir = f'/content/rdd2022/RDD2022_{country}/images/train'
    ann_dir = f'/content/rdd2022/RDD2022_{country}/annotations/xmls'
    if not os.path.exists(img_dir):
        print(f'Skipping {country} — directory not found')
        continue
    imgs = sorted(glob.glob(f'{img_dir}/*.jpg'))
    for img_path in imgs:
        stem = Path(img_path).stem
        xml_path = f'{ann_dir}/{stem}.xml'
        if not os.path.exists(xml_path):
            continue
        from PIL import Image as PILImage
        try:
            with PILImage.open(img_path) as im:
                w, h = im.size
        except:
            continue
        yolo_lines = convert_voc_to_yolo(xml_path, w, h)
        if not yolo_lines:
            continue
        all_samples.append((img_path, yolo_lines, stem))
    print(f'{country}: found {len([s for s in all_samples])} valid samples so far')

# 80/20 train/val split
import random
random.seed(42)
random.shuffle(all_samples)
split = int(len(all_samples) * 0.8)
train_samples = all_samples[:split]
val_samples = all_samples[split:]

def write_split(samples, split_name):
    for img_path, yolo_lines, stem in samples:
        shutil.copy(img_path, f'/content/dataset/images/{split_name}/{stem}.jpg')
        with open(f'/content/dataset/labels/{split_name}/{stem}.txt', 'w') as f:
            f.write('\n'.join(yolo_lines))

write_split(train_samples, 'train')
write_split(val_samples, 'val')

print(f'\nDataset ready:')
print(f'  Train: {len(train_samples)} images')
print(f'  Val:   {len(val_samples)} images')

## Step 5 — Create dataset YAML config

In [ ]:
yaml_content = """path: /content/dataset
train: images/train
val: images/val

nc: 4
names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: pothole
"""

with open('/content/rdd2022.yaml', 'w') as f:
    f.write(yaml_content)

print('Dataset YAML created:')
print(yaml_content)

## Step 6 — Train YOLOv8s

Using `yolov8s` (small) instead of `yolov8n` (nano) for significantly better accuracy.
100 epochs with cosine LR decay and early stopping.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/rdd2022.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    name='road_damage',
    project='/content/runs',
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print('\nTraining complete!')
print(f'Best weights: /content/runs/road_damage/weights/best.pt')

## Step 7 — Evaluate on validation set

In [ ]:
best_model = YOLO('/content/runs/road_damage/weights/best.pt')
metrics = best_model.val(data='/content/rdd2022.yaml', imgsz=640)

print('\n=== Final Validation Metrics ===')
print(f'Precision:  {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)')
print(f'Recall:     {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)')
print(f'mAP@50:     {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)')
print(f'mAP@50-95:  {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)')
print()
print('Per-class mAP@50:')
classes = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
for cls, ap in zip(classes, metrics.box.ap50):
    print(f'  {cls}: {ap*100:.1f}%')

## Step 8 — Download the model

Downloads `road_damage_best.pt` to your local machine.  
Replace `backend/models/road_damage_best.pt` in your repo with this file.

In [ ]:
import shutil
shutil.copy('/content/runs/road_damage/weights/best.pt', '/content/road_damage_best.pt')

from google.colab import files
files.download('/content/road_damage_best.pt')
print('Download started. Save as: backend/models/road_damage_best.pt')

## Step 9 — (Optional) Push model directly to GitHub

Replace `YOUR_TOKEN` with a GitHub PAT with repo write scope.  
This pushes the new model directly to your repo.

In [ ]:
# OPTIONAL — fill in token and run
GITHUB_TOKEN = ''  # your GitHub PAT
REPO = 'manumanoj0010/georoad-inspector'
BRANCH = 'development'

if GITHUB_TOKEN:
    import subprocess
    subprocess.run(['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', '/content/repo'], check=True)
    shutil.copy('/content/road_damage_best.pt', '/content/repo/backend/models/road_damage_best.pt')
    subprocess.run(['git', '-C', '/content/repo', 'config', 'user.email', 'training@georoad.ai'])
    subprocess.run(['git', '-C', '/content/repo', 'config', 'user.name', 'GeoRoad Training'])
    subprocess.run(['git', '-C', '/content/repo', 'add', 'backend/models/road_damage_best.pt'])
    subprocess.run(['git', '-C', '/content/repo', 'commit', '-m', 'Update model: YOLOv8s 100 epochs RDD2022'])
    subprocess.run(['git', '-C', '/content/repo', 'push', 'origin', BRANCH])
    print('Pushed to GitHub!')
else:
    print('Skipped — no token provided. Use Step 8 to download manually.')